In [0]:
CREATE OR REPLACE TABLE workspace.climate_gold.district_risk_score
COMMENT 'Monthly composite drought risk score (0-100) and tier per sub-county'
AS
WITH vulnerability AS (
  -- low soil organic carbon = soil holds less water = more vulnerable (0 = best, 1 = worst)
  SELECT district_id, soil_imputed,
         PERCENT_RANK() OVER (ORDER BY soc_gkg DESC) AS soil_vulnerability
  FROM workspace.climate_silver.district_soil
),
severity AS (
  -- convert anomalies into 0-1 severity: 0 = normal or better, 1 = extreme (2 std devs)
  SELECT a.*, v.soil_vulnerability, v.soil_imputed,
         LEAST(GREATEST(-a.rain_z, 0), 2) / 2 AS rain_severity,
         LEAST(GREATEST(-a.soil_z, 0), 2) / 2 AS soil_severity,
         LEAST(GREATEST( a.heat_z, 0), 2) / 2 AS heat_severity
  FROM workspace.climate_gold.district_anomalies a
  JOIN vulnerability v USING (district_id)
),
scored AS (
  SELECT *,
         ROUND(100 * (0.40 * COALESCE(rain_severity, 0)
                    + 0.25 * COALESCE(soil_severity, 0)
                    + 0.20 * COALESCE(heat_severity, 0)
                    + 0.15 * soil_vulnerability), 1) AS risk_score
  FROM severity
)
SELECT district_id,
       month                         AS as_of_month,
       rain_3m_mm, rain_3m_normal_mm, rain_pct_of_normal,
       rain_z, soil_z, heat_z,
       ROUND(rain_severity, 2)       AS rain_severity,
       ROUND(soil_severity, 2)       AS soil_severity,
       ROUND(heat_severity, 2)       AS heat_severity,
       ROUND(soil_vulnerability, 2)  AS soil_vulnerability,
       soil_imputed,
       risk_score,
       CASE WHEN risk_score >= 75 THEN 'High'
            WHEN risk_score >= 55 THEN 'Elevated'
            WHEN risk_score >= 35 THEN 'Watch'
            ELSE 'Low' END           AS risk_tier
FROM scored;

In [0]:
CREATE OR REPLACE VIEW workspace.climate_gold.district_risk_latest
COMMENT 'Most recent risk score per sub-county, with names and coordinates'
AS
SELECT d.county_name, d.district_name, d.lat, d.lon, r.*
FROM workspace.climate_gold.district_risk_score r
JOIN workspace.climate_silver.districts d USING (district_id)
WHERE r.as_of_month = (SELECT MAX(as_of_month) FROM workspace.climate_gold.district_risk_score);

In [0]:
SELECT as_of_month, risk_tier, COUNT(*) AS sub_counties
FROM workspace.climate_gold.district_risk_score
WHERE as_of_month IN ('2019-12-01', '2022-12-01')
GROUP BY as_of_month, risk_tier
ORDER BY as_of_month, risk_tier;

In [0]:
SELECT as_of_month,
       ROUND(100 * AVG(CASE WHEN risk_tier IN ('High', 'Elevated') THEN 1 ELSE 0 END), 1)
         AS pct_high_or_elevated
FROM workspace.climate_gold.district_risk_score
WHERE as_of_month >= '2005-01-01'
GROUP BY as_of_month
ORDER BY as_of_month;

In [0]:
SELECT d.county_name,
       COUNT(*)                                                  AS sub_counties,
       ROUND(AVG(r.risk_score), 1)                               AS avg_score,
       SUM(CASE WHEN r.risk_tier = 'High' THEN 1 ELSE 0 END)     AS high,
       ROUND(AVG(r.rain_pct_of_normal), 0)                       AS rain_pct_of_normal
FROM workspace.climate_gold.district_risk_score r
JOIN workspace.climate_silver.districts d USING (district_id)
WHERE r.as_of_month = '2022-12-01'
GROUP BY d.county_name
ORDER BY avg_score DESC
LIMIT 15;